<a href="https://colab.research.google.com/github/Ash-cake8/Current-Weather-Conditions-webscraping/blob/main/Order-Financial-SQLlite%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import os
from google.colab import drive

drive.mount('/content/drive')

file_name = "SQL1.sql"
target_path = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if file_name in files:
        target_path = os.path.join(root, file_name)
        break

if target_path:
    print(f"Done: {target_path}")

    connection = sqlite3.connect("retail_sales.db")
    connection.execute("PRAGMA foreign_keys = ON")
    cursor = connection.cursor()

    with open(target_path, 'r', encoding='utf-8') as file:
        sql_script = file.read()

    cursor.executescript(sql_script)
    connection.commit()

    cursor.execute("SELECT COUNT(*) FROM orders")
    print("Succeed:", cursor.fetchone()[0])
else:
    print("sql not found")


Mounted at /content/drive
Done: /content/drive/MyDrive/SQL1.sql
Succeed: 19


In [ ]:
# Task B1: Order Financial Summary
import pandas as pd
query_b1 = """
SELECT
    o.order_id,
    o.order_date,
    o.order_status,
    c.customer_name,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS merchandise_subtotal,
    ROUND(SUM(oi.quantity * oi.unit_price * oi.discount), 2) AS discount_amount,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 2) AS merchandise_total,
    o.shipping_fee,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) + o.shipping_fee, 2) AS final_order_total
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY o.order_id, o.order_date, o.order_status, c.customer_name, o.shipping_fee
ORDER BY final_order_total DESC;
"""

df_b1 = pd.read_sql_query(query_b1, connection)
df_b1

,order_id,order_date,order_status,customer_name,merchandise_subtotal,discount_amount,merchandise_total,shipping_fee,final_order_total
0,1010,2025-04-04,Completed,Smart Office,138050.0,19615.0,118435.0,350,118785.0
1,1004,2025-02-02,Completed,Bright Academy,122400.0,13938.0,108462.0,300,108762.0
2,1019,2025-08-09,Pending,Smart Office,74100.0,6885.0,67215.0,190,67405.0
3,1013,2025-05-03,Completed,Bright Academy,62300.0,5420.0,56880.0,260,57140.0
4,1015,2025-06-08,Completed,Future Lab,61700.0,6510.0,55190.0,310,55500.0
5,1001,2025-01-05,Completed,Alpha Solutions,54750.0,4370.0,50380.0,250,50630.0
6,1009,2025-03-20,Completed,Alpha Solutions,47000.0,4396.0,42604.0,200,42804.0
7,1007,2025-03-01,Completed,Future Lab,42200.0,3514.0,38686.0,280,38966.0
8,1014,2025-05-19,Completed,Khaled Emad,34300.0,2351.0,31949.0,130,32079.0
9,1006,2025-02-15,Pending,Khaled Emad,25400.0,1225.0,24175.0,140,24315.0


In [ ]:
# Task B2: Quarterly Customer Summary
query_b2 = """
SELECT
    c.customer_name,
    CASE
        WHEN strftime('%m', o.order_date) BETWEEN '01' AND '03' THEN strftime('%Y', o.order_date) || '-Q1'
        WHEN strftime('%m', o.order_date) BETWEEN '04' AND '06' THEN strftime('%Y', o.order_date) || '-Q2'
        WHEN strftime('%m', o.order_date) BETWEEN '07' AND '09' THEN strftime('%Y', o.order_date) || '-Q3'
        ELSE strftime('%Y', o.order_date) || '-Q4'
    END AS period,
    COUNT(DISTINCT o.order_id) AS completed_order_count,
    SUM(oi.quantity) AS total_items_purchased,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) + SUM(DISTINCT o.shipping_fee), 2) AS total_completed_spending
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'Completed'
GROUP BY c.customer_id, c.customer_name, period
HAVING COUNT(DISTINCT o.order_id) >= 2
ORDER BY period ASC, total_completed_spending DESC;
"""

df_b2 = pd.read_sql_query(query_b2, connection)
df_b2

,customer_name,period,completed_order_count,total_items_purchased,total_completed_spending
0,Alpha Solutions,2025-Q1,2,12,93434.0


In [ ]:
# Task B3: Unsold and Low-Selling Products
query_b3 = """
SELECT
    p.product_name,
    cat.category_name,
    p.stock_quantity,
    COALESCE(SUM(CASE WHEN o.order_status = 'Completed' THEN oi.quantity ELSE 0 END), 0) AS completed_quantity_sold,
    ROUND(p.stock_quantity * p.unit_price, 2) AS stock_value
FROM products p
JOIN categories cat ON p.category_id = cat.category_id
LEFT JOIN order_items oi ON p.product_id = oi.product_id
LEFT JOIN orders o ON oi.order_id = o.order_id
GROUP BY p.product_id, p.product_name, cat.category_name, p.stock_quantity, p.unit_price
HAVING completed_quantity_sold < 3
ORDER BY
    completed_quantity_sold ASC,
    p.product_name ASC;
"""

df_b3 = pd.read_sql_query(query_b3, connection)
df_b3

,product_name,category_name,stock_quantity,completed_quantity_sold,stock_value
0,Laptop Stand,Accessories,55,0,49500.0


In [ ]:
# Task B4: City-Level Sales Analysis
query_b4 = """
SELECT
    c.city,
    COUNT(DISTINCT c.customer_id) AS registered_customers,
    COUNT(DISTINCT CASE WHEN o.order_status = 'Completed' THEN c.customer_id END) AS purchasing_customers,
    COUNT(DISTINCT CASE WHEN o.order_status = 'Completed' THEN o.order_id END) AS total_completed_orders,
    ROUND(COALESCE(SUM(CASE WHEN o.order_status = 'Completed' THEN oi.quantity * oi.unit_price * (1 - oi.discount) END), 0) +
          COALESCE(SUM(DISTINCT CASE WHEN o.order_status = 'Completed' THEN o.shipping_fee END), 0), 2) AS total_completed_revenue,
    ROUND(
        CASE
            WHEN COUNT(DISTINCT CASE WHEN o.order_status = 'Completed' THEN c.customer_id END) > 0
            THEN (COALESCE(SUM(CASE WHEN o.order_status = 'Completed' THEN oi.quantity * oi.unit_price * (1 - oi.discount) END), 0) +
                  COALESCE(SUM(DISTINCT CASE WHEN o.order_status = 'Completed' THEN o.shipping_fee END), 0)) /
                 COUNT(DISTINCT CASE WHEN o.order_status = 'Completed' THEN c.customer_id END)
            ELSE 0
        END, 2
    ) AS avg_spending_per_purchasing_customer
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
LEFT JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.city
ORDER BY total_completed_revenue DESC;
"""

df_b4 = pd.read_sql_query(query_b4, connection)
df_b4

,city,registered_customers,purchasing_customers,total_completed_orders,total_completed_revenue,avg_spending_per_purchasing_customer
0,Cairo,3,3,7,288269.5,96089.83
1,Tanta,1,1,1,118785.0,118785.00
2,Alexandria,2,1,2,94466.0,94466.00
3,Giza,2,2,3,44271.5,22135.75
4,Mansoura,2,1,1,2830.0,2830.00


In [ ]:
# Task: Discount Impact by Customer Type
query_discount_impact = """
SELECT
    c.customer_type,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS gross_sales,
    ROUND(SUM(oi.quantity * oi.unit_price * oi.discount), 2) AS total_discount_amount,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 2) AS net_merchandise_sales,
    ROUND(
        CASE
            WHEN SUM(oi.quantity * oi.unit_price) > 0
            THEN (SUM(oi.quantity * oi.unit_price * oi.discount) * 100.0) / SUM(oi.quantity * oi.unit_price)
            ELSE 0
        END, 2
    ) AS effective_discount_rate
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'Completed'
GROUP BY c.customer_type
ORDER BY effective_discount_rate DESC;
"""

df_discount_impact = pd.read_sql_query(query_discount_impact, connection)
df_discount_impact

,customer_type,gross_sales,total_discount_amount,net_merchandise_sales,effective_discount_rate
0,Corporate,552400.0,59864.5,492535.5,10.84
1,Premium,46550.0,2638.5,43911.5,5.67
2,Regular,9400.0,0.0,9400.0,0.00


In [ ]:
query_employee_revenue = """
WITH EmployeeRevenue AS (
    SELECT
        e.employee_id,
        e.employee_name,
        e.department,
        COALESCE(SUM(CASE WHEN o.order_status = 'Completed' THEN oi.quantity * oi.unit_price * (1 - oi.discount) END), 0) AS employee_revenue
    FROM employees e
    LEFT JOIN orders o ON e.employee_id = o.employee_id
    LEFT JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY e.employee_id, e.employee_name, e.department),
DepartmentAverage AS (
    SELECT
        department,
        AVG(employee_revenue) AS dept_avg_revenue
    FROM EmployeeRevenue
    GROUP BY department)
SELECT
    er.employee_name,
    er.department,
    ROUND(er.employee_revenue, 2) AS employee_revenue,
    ROUND(da.dept_avg_revenue, 2) AS dept_avg_revenue,
    ROUND(er.employee_revenue - da.dept_avg_revenue, 2) AS revenue_diff
FROM EmployeeRevenue er
JOIN DepartmentAverage da ON er.department = da.department
WHERE er.employee_revenue > da.dept_avg_revenue
ORDER BY er.department ASC, revenue_diff DESC;
"""

df_result = pd.read_sql_query(query_employee_revenue, connection)
df_result

,employee_name,department,employee_revenue,dept_avg_revenue,revenue_diff
0,Mariam Samir,Online Sales,62935.0,33247.5,29687.5
1,Nora Hassan,Sales,210601.0,119838.0,90763.0
2,Salma Tarek,Sales,183015.0,119838.0,63177.0


In [ ]:
# Task B7: Frequently Purchased Product Pairs
query_b7 = """
SELECT
    prod1.product_name AS product_1_name,
    prod2.product_name AS product_2_name,
    COUNT(DISTINCT oi1.order_id) AS pair_order_count,
    SUM(oi1.quantity + oi2.quantity) AS combined_quantity_sold
FROM order_items oi1
JOIN order_items oi2 ON oi1.order_id = oi2.order_id AND oi1.product_id < oi2.product_id
JOIN orders o ON oi1.order_id = o.order_id
JOIN products prod1 ON oi1.product_id = prod1.product_id
JOIN products prod2 ON oi2.product_id = prod2.product_id
WHERE o.order_status = 'Completed'
GROUP BY oi1.product_id, oi2.product_id, prod1.product_name, prod2.product_name
HAVING COUNT(DISTINCT oi1.order_id) >= 2
ORDER BY pair_order_count DESC, combined_quantity_sold DESC;
"""

df_b7 = pd.read_sql_query(query_b7, connection)
df_b7

,product_1_name,product_2_name,pair_order_count,combined_quantity_sold
0,Vision 24 Monitor,Eight-Port Switch,3,18
1,Vision 27 Monitor,Dual-Band Router,2,12
2,ProBook X15,Vision 27 Monitor,2,9


In [ ]:
# Task B8: Cumulative Monthly Revenue
query_b8_simple = """
SELECT
    strftime('%Y-%m', o.order_date) AS sales_month,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) + SUM(DISTINCT o.shipping_fee), 2) AS monthly_revenue,
    ROUND(SUM(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) + SUM(DISTINCT o.shipping_fee)) OVER (ORDER BY strftime('%Y-%m', o.order_date)), 2) AS cumulative_revenue,
    ROUND((SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) + SUM(DISTINCT o.shipping_fee)) * 100.0 / SUM(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) + SUM(DISTINCT o.shipping_fee)) OVER (), 2) AS pct_of_total_revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'Completed'
GROUP BY sales_month
ORDER BY sales_month ASC;
"""

df_b8 = pd.read_sql_query(query_b8_simple, connection)
df_b8

,sales_month,monthly_revenue,cumulative_revenue,pct_of_total_revenue
0,2025-01,57707.5,57707.5,10.52
1,2025-02,111592.0,169299.5,20.34
2,2025-03,85410.0,254709.5,15.57
3,2025-04,123900.0,378609.5,22.58
4,2025-05,89219.0,467828.5,16.26
5,2025-06,55500.0,523328.5,10.12
6,2025-07,25293.5,548622.0,4.61
